# Linear Regression

**难度:** Medium

用三种方式实现线性回归：闭式解、梯度下降和 nn.Linear。

本题涵盖拟合线性模型的基本方法，从正规方程到基于 autograd 的训练。

**签名:** `LinearRegression()`（类）

**方法:**
- `closed_form(X, y) -> (w, b)` — 通过正规方程求解
- `gradient_descent(X, y, lr, steps) -> (w, b)` — 手动梯度更新
- `nn_linear(X, y, lr, steps) -> (w, b)` — PyTorch autograd 训练循环

**约束:**
- 三种方法应收敛到相近的权重
- 闭式解不应使用 autograd
- X 为 (N, D)，y 为 (N,)，返回 w (D,) 和 b（标量）

**提示:**
closed_form：X 增加全 1 列 → `lstsq(X_aug, y)` → 拆分 `w, b`
gradient_descent：`grad_w = (2/N) * X.T @ (X@w+b - y)`，`w -= lr*grad_w`
nn_linear：`nn.Linear(D,1)` + `MSELoss` + `optimizer.step()` 循环

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ SOLUTION

class LinearRegression:
    # ---- 方法 1: 闭式解（正规方程）----
    def closed_form(self, X: torch.Tensor, y: torch.Tensor):
        # 目标：求解 y = Xw + b 的最小二乘解
        # 技巧：将偏置 b 吸收进权重，令 X_aug = [X | 1]
        # 则 y = X_aug @ theta，其中 theta = [w, b]
        N, D = X.shape

        # 在 X 末尾拼接一列全 1
        X_aug = torch.cat([X, torch.ones(N, 1)], dim=1)  # (N, D+1)

        # 用最小二乘求解：theta = (X^T X)^{-1} X^T y
        # torch.linalg.lstsq 比手写逆矩阵更数值稳定
        theta = torch.linalg.lstsq(X_aug, y).solution  # (D+1,)

        # 拆分出权重和偏置
        w = theta[:D]    # (D,)
        b = theta[D]     # scalar
        return w.detach(), b.detach()

    # ---- 方法 2: 梯度下降（手动推导梯度）----
    def gradient_descent(self, X: torch.Tensor, y: torch.Tensor,
                         lr: float = 0.01, steps: int = 1000):
        N, D = X.shape
        w = torch.zeros(D)       # 初始化权重为零
        b = torch.tensor(0.0)    # 初始化偏置为零

        for _ in range(steps):
            # 前向传播：预测值
            pred = X @ w + b          # (N,)
            error = pred - y           # (N,) 预测误差

            # 手动计算 MSE 梯度
            # MSE = (1/N) * ||pred - y||^2
            # dMSE/dw = (2/N) * X^T @ (pred - y)  → shape: (D,)
            # dMSE/db = (2/N) * sum(pred - y)      → shape: scalar
            grad_w = (2.0 / N) * (X.T @ error)  # (D,)
            grad_b = (2.0 / N) * error.sum()     # scalar

            # 梯度下降更新（不是 inplace，而是重新赋值）
            w = w - lr * grad_w
            b = b - lr * grad_b

        return w, b

    # ---- 方法 3: nn.Linear + autograd ----
    def nn_linear(self, X: torch.Tensor, y: torch.Tensor,
                  lr: float = 0.01, steps: int = 1000):
        N, D = X.shape
        # 创建线性层：自动初始化 w 和 b
        layer = nn.Linear(D, 1)
        optimizer = torch.optim.SGD(layer.parameters(), lr=lr)
        loss_fn = nn.MSELoss()

        for _ in range(steps):
            optimizer.zero_grad()           # 清零梯度
            pred = layer(X).squeeze(-1)     # (N,1) → (N,)
            loss = loss_fn(pred, y)         # MSE 损失
            loss.backward()                 # 反向传播计算梯度
            optimizer.step()                # 更新参数

        # 提取训练好的参数
        w = layer.weight.data.squeeze(0)  # (1, D) → (D,)
        b = layer.bias.data.squeeze(0)    # (1,) → scalar
        return w, b

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check("linear_regression")

## 📝 核心思路总结

1. **闭式解（正规方程）**：一步到位，但 O(D³) 复杂度，大维度不可行
2. **梯度下降**：手动推导梯度，理解 MSE 损失的导数 = (2/N) * X^T * error
3. **nn.Linear + autograd**：PyTorch 自动求梯度，最实用但面试要能手写